# Chronic Kidney Disease Prediction
Using **Bagging** (Bagging, Random Forest) and **Boosting** (AdaBoost, Gradient Boosting, XGBoost) algorithms.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('kidney_disease.csv')
df.head()

In [ ]:
print(df.shape)
df.info()

## 3. Data Cleaning

In [ ]:
# Drop id column
df = df.drop('id', axis=1)

# Strip stray tabs/spaces and turn '?' into NaN in text columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype(str).str.strip().replace({'?': np.nan, 'nan': np.nan, '': np.nan})

df['classification'].value_counts()

In [ ]:
# pcv, wc, rc should be numeric
for col in ['pcv', 'wc', 'rc']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

df.dtypes

In [ ]:
# Missing values per column
df.isnull().sum().sort_values(ascending=False)

In [ ]:
# Impute: median for numeric, mode for categorical
num_cols = df.select_dtypes(include='number').columns
cat_cols = df.select_dtypes(include='object').columns

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print('Remaining missing values:', df.isnull().sum().sum())

## 4. Encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Target: ckd = 1, notckd = 0
df['classification'] = df['classification'].map({'ckd': 1, 'notckd': 0})

# Encode remaining categorical features
le = LabelEncoder()
for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

df.head()

## 5. Exploratory Data Analysis

In [ ]:
sns.countplot(x='classification', data=df)
plt.title('Class Distribution (0 = Not CKD, 1 = CKD)')
plt.show()

In [ ]:
plt.figure(figsize=(12, 9))
sns.heatmap(df.corr(), cmap='coolwarm', center=0)
plt.title('Correlation Heatmap')
plt.show()

## 6. Train / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('classification', axis=1)
y = df['classification']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape, X_test.shape)

## 7. Evaluation Helper

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report)

results = {}
predictions = {}

def evaluate(name, model):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    predictions[name] = pred
    results[name] = {
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1-score': f1_score(y_test, pred)
    }
    print(f'--- {name} ---')
    print(classification_report(y_test, pred, target_names=['Not CKD', 'CKD']))

## 8. Bagging Algorithms

In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag = BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=42)
evaluate('Bagging', bag)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
evaluate('Random Forest', rf)

## 9. Boosting Algorithms

In [ ]:
from sklearn.ensemble import AdaBoostClassifier

ada = AdaBoostClassifier(n_estimators=100, random_state=42)
evaluate('AdaBoost', ada)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
evaluate('Gradient Boosting', gb)

In [ ]:
# XGBoost (install with: pip install xgboost)
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, eval_metric='logloss', random_state=42)
evaluate('XGBoost', xgb)

## 10. Model Comparison

In [ ]:
results_df = pd.DataFrame(results).T.round(4)
results_df.sort_values('Accuracy', ascending=False)

In [ ]:
results_df['Accuracy'].plot(kind='bar', figsize=(8, 4), ylim=(0.9, 1.01))
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy')
plt.xticks(rotation=30)
plt.show()

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, len(predictions), figsize=(4 * len(predictions), 3.5))
for ax, (name, pred) in zip(axes, predictions.items()):
    sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.show()

## 11. Feature Importance

In [ ]:
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
imp.head(10).plot(kind='barh', figsize=(7, 4)).invert_yaxis()
plt.title('Top 10 Features (Random Forest)')
plt.show()

## 12. Cross-Validation (Optional)

In [ ]:
from sklearn.model_selection import cross_val_score

models = {'Bagging': bag, 'Random Forest': rf, 'AdaBoost': ada,
          'Gradient Boosting': gb, 'XGBoost': xgb}
for name, m in models.items():
    scores = cross_val_score(m, X, y, cv=5, scoring='accuracy')
    print(f'{name:18s} mean = {scores.mean():.4f}  std = {scores.std():.4f}')

## 13. Conclusion
- Both bagging and boosting ensembles predict CKD with high accuracy.
- Bagging methods (Bagging, Random Forest) reduce variance by averaging many trees trained on bootstrap samples.
- Boosting methods (AdaBoost, Gradient Boosting, XGBoost) reduce bias by training models sequentially on previous errors.
- Compare the table above to pick the best model; features such as `hemo`, `sg`, `al`, `sc` usually matter most.